# 11 — Drift Monitor

Detect statistical drift between reference and current DataFrames using KS tests, Chi-squared, and Population Stability Index (PSI).

In [ ]:
from sqllocks_spindle import Spindle
from sqllocks_spindle.inference.tier3_research import DriftMonitor
import importlib, numpy as np
from sqllocks_spindle.cli import _get_domain_registry

spindle = Spindle()
reg = _get_domain_registry()
mod, cls, _ = reg['retail']
domain = getattr(importlib.import_module(mod), cls)(schema_mode='3nf')

ref_result = spindle.generate(domain=domain, scale='small', seed=42)
cur_result = spindle.generate(domain=domain, scale='small', seed=99)
table = next(iter(ref_result.tables))
print(f'Monitoring drift for: {table}')

In [ ]:
monitor = DriftMonitor(pvalue_threshold=0.05, psi_threshold=0.2)
report = monitor.compare(ref_result.tables[table], cur_result.tables[table])

print(f'Overall drift score: {report.overall_drift_score:.4f}')
print(f'Drift fraction: {report.drift_fraction:.1%}')
print(f'Drifted columns: {report.drifted_columns}')

In [ ]:
# Per-column detail
print('\n--- Per-Column Drift Results ---')
print(f'{"Column":<25} {"Method":<8} {"Stat":>10} {"p-value":>10} {"Drifted?"}')
print('-' * 65)
for col, r in sorted(report.columns.items(), key=lambda x: -x[1].drift_score):
    p_str = f'{r.p_value:.4f}' if r.p_value is not None else '  N/A'
    stat_str = f'{r.test_statistic:.4f}' if r.test_statistic is not None else '  N/A'
    drifted = 'YES' if r.is_drifted else 'no'
    print(f'{col:<25} {r.method:<8} {stat_str:>10} {p_str:>10}  {drifted}')

In [ ]:
# Simulate intentional drift and detect it
import pandas as pd, numpy as np
ref_df = ref_result.tables[table].copy()
drifted_df = cur_result.tables[table].copy()

# Shift all numeric columns by 3 standard deviations
for col in drifted_df.select_dtypes(include='number').columns[:3]:
    shift = drifted_df[col].std() * 3
    drifted_df[col] = drifted_df[col] + shift

drift_report = monitor.compare(ref_df, drifted_df)
print(f'\nArtificial drift — drifted columns: {drift_report.drifted_columns}')